In [26]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent  # go from notebooks/ -> repo root
print("ROOT =", ROOT)

clean  = list((ROOT/"data/generated/train/clean_audio").glob("*.npy"))
noisy  = list((ROOT/"data/generated/train/noisy_audio").glob("*.npy"))
labels = list((ROOT/"data/generated/train/labels").glob("*_y.npy"))

print(len(clean), len(noisy), len(labels))

ROOT = d:\Projects\CS6140\Noise-Robust-Voice-Activity-Detection
2000 2000 2000


In [27]:
import json
import numpy as np
from pathlib import Path
import sys

# --- setup paths (since notebook is in notebooks/) ---
ROOT = Path.cwd().parent
NOISE_DIR = ROOT / "src" / "03_add_noise"
sys.path.insert(0, str(NOISE_DIR))

from utils_noise import rms, speech_mask_from_intervals  # your functions

# --- load one manifest row ---
manifest_path = ROOT / "data" / "generated" / "train" / "manifests" / "train_noisy_manifest.jsonl"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest_row = json.loads(next(f))   # first example; you can randomize later

# --- build file paths from manifest (paths are relative to split folder) ---
split_dir = ROOT / "data" / "generated" / "train"
clean_path = split_dir / manifest_row["clean_audio_path"]
noisy_path = split_dir / manifest_row["noisy_audio_path"]

x_clean = np.load(clean_path)
x_noisy = np.load(noisy_path)

n = x_noisy - x_clean
mask = speech_mask_from_intervals(len(x_clean), manifest_row["speech_intervals"])

snr = 20 * np.log10(rms(x_clean[mask]) / (rms(n[mask]) + 1e-12))

print("ex_id:", manifest_row["ex_id"])
print("target SNR:", manifest_row["noise"]["snr_db_target"])
print("actual  SNR:", float(snr))

ex_id: train_0000000
target SNR: 10.0
actual  SNR: 10.000000156825967


In [28]:
import numpy as np
x = np.load(noisy[0])
print(x.dtype, x.shape, x.min(), x.max())

float32 (236536,) -0.7067396 0.7156863


In [32]:
import numpy as np
import json
from pathlib import Path

ROOT = Path.cwd().parent
split_dir = ROOT / "data" / "generated" / "train"
manifest_path = split_dir / "manifests" / "train_noisy_manifest.jsonl"

# load one row
with open(manifest_path, "r", encoding="utf-8") as f:
    row = json.loads(next(f))

x_clean = np.load(split_dir / row["clean_audio_path"])
x_noisy = np.load(split_dir / row["noisy_audio_path"])

mask = speech_mask_from_intervals(len(x_clean), row["speech_intervals"])

g = float(row["noise"].get("clip_gain", 1.0))  # if no clipping happened, g=1

n_eff = (x_noisy / g) - x_clean  # <-- key line

snr_actual = 20*np.log10(rms(x_clean[mask]) / (rms(n_eff[mask]) + 1e-12))

print("target:", row["noise"]["snr_db_target"])
print("actual :", float(snr_actual))
print("clip_gain:", g)

target: 10.0
actual : 10.000000156825967
clip_gain: 1.0


In [33]:
import numpy as np, json
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().parent
split_dir = ROOT / "data" / "generated" / "train"
manifest_path = split_dir / "manifests" / "train_noisy_manifest.jsonl"

targets, actuals, gains = [], [], []

with open(manifest_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 200:
            break
        row = json.loads(line)

        x_clean = np.load(split_dir / row["clean_audio_path"])
        x_noisy = np.load(split_dir / row["noisy_audio_path"])
        mask = speech_mask_from_intervals(len(x_clean), row["speech_intervals"])

        g = float(row["noise"].get("clip_gain", 1.0))
        n_eff = (x_noisy / g) - x_clean

        snr_actual = 20*np.log10(rms(x_clean[mask]) / (rms(n_eff[mask]) + 1e-12))

        targets.append(float(row["noise"]["snr_db_target"]))
        actuals.append(float(snr_actual))
        gains.append(g)

print("Target bucket counts:", Counter(targets))
print("Mean |actual-target|:", float(np.mean(np.abs(np.array(actuals) - np.array(targets)))))
print("Clip applied count:", sum(1 for g in gains if g < 0.999999))

Target bucket counts: Counter({10.0: 47, -5.0: 44, 5.0: 38, 0.0: 36, 20.0: 35})
Mean |actual-target|: 1.9348157168718152e-07
Clip applied count: 40


In [34]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent
manifest_path = ROOT/"data/generated/train/manifests/train_noisy_manifest.jsonl"

c = Counter()
with open(manifest_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 200:
            break
        r = json.loads(line)
        c[r["noise"]["noise_type"]] += 1
print(c)

Counter({'noise': 103, 'music': 60, 'babble': 37})


In [35]:
import json, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
split_dir = ROOT/"data/generated/train"
manifest_path = split_dir/"manifests/train_noisy_manifest.jsonl"

sr = 16000
L = int(0.025*sr)
H = int(0.01*sr)

with open(manifest_path, "r", encoding="utf-8") as f:
    for _ in range(5):
        r = json.loads(next(f))
        x = np.load(split_dir / r["clean_audio_path"])
        y = np.load(split_dir / r["labels_path"])
        expected = (len(x) - L)//H + 1
        print(r["ex_id"], len(y), expected)

train_0000000 1476 1476
train_0000001 2083 2083
train_0000002 2672 2672
train_0000003 2157 2157
train_0000004 1554 1554
